# ⚡ Module 04 — BAML & Pydantic: Type-Safe LLM Pipelines

> **Marevlo AI Platform** · All Levels: Beginner → Expert

---
### What you'll learn
- Writing Pydantic models for LLM output validation
- `field_validator` and `model_validator` for custom business rules
- Converting Pydantic models to Anthropic tool schemas
- Retry loops with structured error feedback
- Async batch processing with validation
- BAML syntax, concepts, and generated client patterns
- Production model hierarchy pattern (LLMOutput → Domain → API)

---

In [ ]:
!pip install pydantic pydantic-settings instructor anthropic -q

---
## ⚡ Part 1 — Pydantic Fundamentals
### 🟢 Beginner: Your First Pydantic Model

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import Literal, Optional
import json

# --- Define a typed model ---
class DeviceReport(BaseModel):
    device_id:  str
    risk_level: Literal["LOW", "MED", "HIGH"]
    score:      float = Field(ge=0.0, le=1.0, description="0=normal, 1=critical")
    root_cause: str
    actions:    list[str] = Field(min_length=1)

# ✓ Valid construction
report = DeviceReport(
    device_id  = "JNP-001",
    risk_level = "HIGH",
    score      = 0.91,
    root_cause = "rpd memory leak",
    actions    = ["Restart rpd", "Check BGP peers"]
)
print("✓ Valid report created")
print(f"  device_id: {report.device_id}")
print(f"  risk_level: {report.risk_level}")
print(f"  score: {report.score}")
print(f"  actions: {report.actions}")
print(f"\nJSON: {report.model_dump_json()}")

In [ ]:
# ✗ Multiple validation failures at once
invalid_cases = [
    {
        "name": "Invalid enum + score out of range + empty actions",
        "data": {"device_id": "JNP-001", "risk_level": "CRITICAL",
                 "score": 1.5, "root_cause": "unknown", "actions": []}
    },
    {
        "name": "Missing required field",
        "data": {"device_id": "JNP-001", "score": 0.5, "root_cause": "ok", "actions": ["check"]}
    },
    {
        "name": "Wrong type for score",
        "data": {"device_id": "JNP-001", "risk_level": "LOW",
                 "score": "not-a-number", "root_cause": "ok", "actions": ["check"]}
    },
]

for case in invalid_cases:
    try:
        DeviceReport.model_validate(case["data"])
        print(f"UNEXPECTED PASS: {case['name']}")
    except ValidationError as e:
        errs = e.errors()
        print(f"\n✗ {case['name']} ({len(errs)} errors):")
        for err in errs:
            field = '.'.join(str(x) for x in err['loc'])
            print(f"  [{field}] {err['msg']}")

In [ ]:
# Auto-generated JSON Schema
schema = DeviceReport.model_json_schema()
print("Auto-generated JSON Schema:")
print(json.dumps(schema, indent=2))

### 🟡 Intermediate: Nested Models & Type Coercion

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional
from datetime import datetime

class Metric(BaseModel):
    name:         str
    value:        float
    unit:         str
    is_anomalous: bool

class RemediationStep(BaseModel):
    order:             int = Field(ge=1)
    action:            str = Field(min_length=10)
    estimated_minutes: int = Field(ge=1, le=480)
    requires_downtime: bool = False

class ComprehensiveReport(BaseModel):
    device_id:  str
    timestamp:  datetime        # Will coerce ISO strings
    risk_level: Literal["LOW", "MED", "HIGH"]
    score:      float
    metrics:    list[Metric]
    steps:      list[RemediationStep]
    notes:      Optional[str] = None

# LLMs often return strings for numbers — Pydantic coerces them
llm_output = {
    "device_id":  "JNP-001",
    "timestamp":  "2024-03-15T06:00:00Z",   # str → datetime ✓
    "risk_level": "HIGH",
    "score":      "0.91",                    # str → float ✓
    "metrics": [
        {
            "name": "cpu_util",
            "value": "94.2",                 # str → float ✓
            "unit": "%",
            "is_anomalous": "true"           # str → bool ✓
        }
    ],
    "steps": [
        {
            "order": "1",                    # str → int ✓
            "action": "Restart the routing process daemon (rpd)",
            "estimated_minutes": "5"         # str → int ✓
        }
    ]
}

report = ComprehensiveReport.model_validate(llm_output)
print(f"✓ Type coercion worked:")
print(f"  score: {report.score} (type: {type(report.score).__name__})")
print(f"  timestamp: {type(report.timestamp).__name__}")
print(f"  is_anomalous: {report.metrics[0].is_anomalous} (type: {type(report.metrics[0].is_anomalous).__name__})")
print(f"  step order: {report.steps[0].order} (type: {type(report.steps[0].order).__name__})")

---
## 🔍 Part 2 — Custom Validators
### 🟢 Beginner: field_validator

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Literal, Optional
import re

class DeviceReport(BaseModel):
    device_id:  str
    risk_level: str
    score:      float = Field(ge=0, le=1)

    @field_validator("device_id")
    @classmethod
    def validate_device_id_format(cls, v: str) -> str:
        """Enforce device ID format AND normalize to uppercase."""
        v = v.strip().upper()
        if not re.match(r"^[A-Z]{2,4}-\d{3}$", v):
            raise ValueError(f"Invalid device ID: '{v}'. Expected format: JNP-001")
        return v

    @field_validator("risk_level")
    @classmethod
    def normalize_risk_level(cls, v: str) -> str:
        """Accept case-insensitive input and normalize."""
        normalized = v.strip().upper()
        if normalized not in {"LOW", "MED", "HIGH"}:
            raise ValueError(f"risk_level must be LOW/MED/HIGH, got: '{v}'")
        return normalized

# Test normalization
r1 = DeviceReport(device_id="jnp-001", risk_level="high", score=0.9)
print(f"Normalized device_id: {r1.device_id}")  # JNP-001
print(f"Normalized risk_level: {r1.risk_level}") # HIGH

# Test validation
test_cases = [
    {"device_id": "router-1",  "risk_level": "HIGH", "score": 0.5},  # Bad ID
    {"device_id": "JNP-001",   "risk_level": "EXTREME", "score": 0.9},  # Bad risk
    {"device_id": "JNP-001",   "risk_level": "HIGH", "score": 0.9},  # Valid
]

print("\nValidation tests:")
for case in test_cases:
    try:
        r = DeviceReport.model_validate(case)
        print(f"  ✓ Valid: {r.device_id} → {r.risk_level}")
    except Exception as e:
        print(f"  ✗ Invalid: {e.errors()[0]['msg']}")

In [ ]:
# model_validator: Cross-field business rules

class AlertReport(BaseModel):
    risk_level:         Literal["LOW", "MED", "HIGH"]
    score:              float = Field(ge=0, le=1)
    escalation_contact: Optional[str] = None
    actions:            list[str]
    notify_oncall:      bool = False

    @model_validator(mode="after")
    def high_risk_requires_escalation(self):
        if self.risk_level == "HIGH" and not self.escalation_contact:
            raise ValueError("escalation_contact is required when risk_level is HIGH")
        return self

    @model_validator(mode="after")
    def score_consistent_with_risk(self):
        ranges = {"LOW": (0.0, 0.5), "MED": (0.4, 0.8), "HIGH": (0.7, 1.0)}
        lo, hi = ranges[self.risk_level]
        if not (lo <= self.score <= hi):
            raise ValueError(
                f"score={self.score:.2f} inconsistent with {self.risk_level} risk "
                f"(expected {lo:.1f}–{hi:.1f})"
            )
        return self

    @model_validator(mode="after")
    def auto_set_oncall_for_high_risk(self):
        if self.risk_level == "HIGH":
            self.notify_oncall = True
        return self

# Test cross-field rules
print("Cross-field validation tests:")

# Should fail: HIGH risk without escalation
try:
    AlertReport(risk_level="HIGH", score=0.91, actions=["check"])
except Exception as e:
    print(f"  ✗ HIGH without escalation: {e.errors()[0]['msg']}")

# Should fail: score doesn't match risk level
try:
    AlertReport(risk_level="HIGH", score=0.3, escalation_contact="oncall@marevlo.com", actions=["check"])
except Exception as e:
    print(f"  ✗ Score mismatch: {e.errors()[0]['msg']}")

# Should succeed + auto-set notify_oncall
alert = AlertReport(
    risk_level="HIGH", score=0.91,
    escalation_contact="oncall@marevlo.com",
    actions=["Restart rpd"]
)
print(f"  ✓ Valid HIGH alert, notify_oncall auto-set: {alert.notify_oncall}")

### 🔴 Advanced: Computed Fields & Custom Serializers

In [ ]:
from pydantic import BaseModel, Field, computed_field, field_serializer
from typing import Literal
from datetime import datetime, timezone

class DeviceReport(BaseModel):
    device_id:  str
    risk_level: Literal["LOW", "MED", "HIGH"]
    score:      float
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    metrics:    dict[str, float] = Field(default_factory=dict)

    # Computed fields — derived, not stored in input
    @computed_field
    @property
    def severity_code(self) -> int:
        return {"LOW": 1, "MED": 2, "HIGH": 3}[self.risk_level]

    @computed_field
    @property
    def requires_immediate_action(self) -> bool:
        return self.risk_level == "HIGH" and self.score >= 0.85

    @computed_field
    @property
    def anomalous_metrics(self) -> list[str]:
        return [k for k, v in self.metrics.items() if v > 0.8]

    # Custom serializers — control JSON output format
    @field_serializer("created_at")
    def serialize_dt(self, dt: datetime) -> str:
        return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

    @field_serializer("score")
    def serialize_score(self, v: float) -> str:
        return f"{v:.3f}"  # Always 3 decimal places

report = DeviceReport(
    device_id="JNP-001",
    risk_level="HIGH",
    score=0.91,
    metrics={
        "cpu_util": 0.942,
        "mem_util": 0.673,
        "pfe_errors": 0.950
    }
)

print(f"severity_code: {report.severity_code}")           # 3
print(f"immediate: {report.requires_immediate_action}")   # True
print(f"anomalous: {report.anomalous_metrics}")           # ['cpu_util', 'pfe_errors']
print(f"\nJSON output:")
print(report.model_dump_json(indent=2))

---
## 🤖 Part 3 — LLM Output Validation
### 🟢 Beginner: Pydantic Tool Schema Pipeline

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
import anthropic, json

class DeviceAnalysis(BaseModel):
    device_id:  str = Field(description="Analyzed device identifier")
    risk_level: Literal["LOW", "MED", "HIGH"] = Field(
        description="LOW=monitor only, MED=schedule maintenance, HIGH=immediate action required"
    )
    score:      float = Field(ge=0, le=1, description="0.0=completely normal, 1.0=critical failure imminent")
    root_cause: str   = Field(description="Single-sentence root cause explanation")
    actions:    list[str] = Field(min_length=1, description="Ordered list of remediation steps")

def pydantic_to_anthropic_tool(model: type[BaseModel], name: str, desc: str) -> dict:
    """Convert a Pydantic model to an Anthropic tool definition."""
    schema = model.model_json_schema()
    schema.pop("$defs", None)
    schema.pop("title", None)
    return {"name": name, "description": desc, "input_schema": schema}

client = anthropic.Anthropic()
tool = pydantic_to_anthropic_tool(
    DeviceAnalysis,
    "report_device_analysis",
    "Report the structured findings from device analysis"
)

response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    tools=[tool],
    tool_choice={"type": "tool", "name": "report_device_analysis"},
    messages=[{
        "role": "user",
        "content": "Analyze device JNP-001: CPU=94.2%, memory=67.3%, PFE errors=412/hr, BGP flaps=8"
    }]
)

# Extract and validate in one step
raw = response.content[0].input
analysis = DeviceAnalysis.model_validate(raw)

print(f"Device: {analysis.device_id}")
print(f"Risk: {analysis.risk_level} (score: {analysis.score:.2f})")
print(f"Cause: {analysis.root_cause}")
print(f"Actions ({len(analysis.actions)}):")
for i, action in enumerate(analysis.actions, 1):
    print(f"  {i}. {action}")

### 🟡 Intermediate: Retry Loop with Error Feedback

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import Literal, TypeVar
import anthropic

T = TypeVar('T', bound=BaseModel)

def validated_llm_call(
    model_class: type[T],
    prompt: str,
    tool_name: str = "structured_output",
    max_retries: int = 3
) -> T:
    """Call Claude with automatic Pydantic validation and error-feedback retry."""
    client = anthropic.Anthropic()

    # Build tool from model
    schema = model_class.model_json_schema()
    schema.pop("$defs", None)
    schema.pop("title", None)
    tool_def = {
        "name": tool_name,
        "description": "Return structured output matching the schema exactly",
        "input_schema": schema
    }

    last_errors = []

    for attempt in range(max_retries):
        # Build prompt with error feedback on retry
        current_prompt = prompt
        if last_errors:
            error_detail = "\n".join(f"  - {e}" for e in last_errors)
            current_prompt += f"\n\n⚠️ Previous response had validation errors:\n{error_detail}\nPlease fix ALL errors in your next response."

        response = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=1024,
            tools=[tool_def],
            tool_choice={"type": "tool", "name": tool_name},
            messages=[{"role": "user", "content": current_prompt}]
        )

        raw = response.content[0].input

        try:
            result = model_class.model_validate(raw)
            print(f"  ✓ Validated on attempt {attempt + 1}")
            return result
        except ValidationError as e:
            last_errors = [f"{'.'.join(str(x) for x in err['loc'])}: {err['msg']}" for err in e.errors()]
            print(f"  ✗ Attempt {attempt + 1} failed ({len(last_errors)} errors): {last_errors[0]}")

    raise RuntimeError(f"Failed after {max_retries} attempts. Last errors: {last_errors}")

# Test it
class StrictReport(BaseModel):
    device_id:  str = Field(pattern=r"^[A-Z]{2,4}-\d{3}$")
    risk_level: Literal["LOW", "MED", "HIGH"]
    score:      float = Field(ge=0, le=1)
    actions:    list[str] = Field(min_length=1, max_length=5)

print("Running validated LLM call with retry:")
result = validated_llm_call(
    StrictReport,
    "Analyze device JNP-001: CPU=94.2%, PFE errors=412/hr. Risk assessment required."
)
print(f"Final: {result.device_id} → {result.risk_level} (score: {result.score:.2f})")

### 🔴 Advanced: Instructor Library Integration

In [ ]:
import instructor
import anthropic
from pydantic import BaseModel, Field
from typing import Literal

class DeviceAnalysis(BaseModel):
    device_id:  str
    risk_level: Literal["LOW", "MED", "HIGH"] = Field(
        description="LOW=monitor, MED=schedule maintenance within 48h, HIGH=immediate action"
    )
    score:      float = Field(ge=0, le=1, description="0=completely normal, 1=imminent failure")
    root_cause: str   = Field(description="One clear sentence explaining the root cause")
    top_actions: list[str] = Field(min_length=1, max_length=3,
                                    description="Top 3 priority actions, ordered")

# Instructor patches the client — one extra parameter does everything
client = instructor.from_anthropic(anthropic.Anthropic())

analysis: DeviceAnalysis = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    response_model=DeviceAnalysis,   # ← The magic line
    max_retries=3,                   # ← Auto-retry with error feedback
    messages=[{
        "role": "user",
        "content": """Analyze network device telemetry:
Device: JNP-001
CPU utilization: 94.2% (threshold: 80%)
Memory utilization: 67.3%
PFE errors last hour: 412 (baseline: <10)
BGP session flaps 24hr: 8"""
    }]
)

# analysis is a fully validated, typed Python object
print(f"Device: {analysis.device_id}")
print(f"Risk: {analysis.risk_level} (score: {analysis.score:.2f})")
print(f"Cause: {analysis.root_cause}")
print(f"\nTop actions:")
for i, action in enumerate(analysis.top_actions, 1):
    print(f"  {i}. {action}")

# Type-safe attributes with IDE support
assert isinstance(analysis.score, float)
assert isinstance(analysis.risk_level, str)
assert isinstance(analysis.top_actions, list)

---
## 🏗️ Part 4 — BAML Concepts
### 🟢 Beginner: Understanding BAML Files

In [ ]:
# BAML is a separate DSL — you write .baml files and run:
#   baml-cli generate
# which creates a baml_client/ Python package.

# This cell shows the BAML concepts and how the generated
# code would look, even without BAML installed.

BAML_EXAMPLE = """
// device_analysis.baml

// 1. Define output enum
enum RiskLevel {
  LOW  @description("Normal — monitor only")
  MED  @description("Schedule maintenance within 48 hours")
  HIGH @description("Immediate intervention required")
}

// 2. Define output class
class DeviceReport {
  device_id   string    @description("Device ID e.g. JNP-001")
  risk_level  RiskLevel
  score       float     @description("0.0=normal, 1.0=critical")
  root_cause  string    @description("Single-sentence root cause")
  actions     string[]  @description("Ordered remediation steps")
}

// 3. Configure LLM client
client<llm> Claude {
  provider anthropic
  options {
    model claude-opus-4-5
    max_tokens 1024
  }
}

// 4. Define the LLM function
function AnalyzeDevice(
  device_id: string,
  telemetry: string
) -> DeviceReport {
  client Claude
  prompt #"
    You are a network reliability engineer.
    
    Analyze device {{ device_id }} with this telemetry:
    {{ telemetry }}
    
    {{ ctx.output_format }}
  "#
}

// 5. Write test cases
test HighRiskScenario {
  functions [AnalyzeDevice]
  args {
    device_id "JNP-001"
    telemetry "CPU: 94.2%, PFE errors: 412/hr"
  }
  @check(risk_level == RiskLevel.HIGH, "Critical device should be HIGH")
  @check(score >= 0.8, "High score expected")
}
"""

print("BAML file content:")
print(BAML_EXAMPLE)

In [ ]:
# The generated Python client from BAML would look like:

BAML_GENERATED_PYTHON = """
# This is AUTO-GENERATED by: baml-cli generate
# Do not edit manually!

# baml_client/__init__.py
from baml_client.sync_client import BamlSyncClient

b = BamlSyncClient()

# baml_client/types.py  
from enum import Enum
class RiskLevel(str, Enum):
    LOW  = "LOW"
    MED  = "MED"
    HIGH = "HIGH"

from pydantic import BaseModel
class DeviceReport(BaseModel):
    device_id:  str
    risk_level: RiskLevel
    score:      float
    root_cause: str
    actions:    list[str]

# Usage (after baml-cli generate):
# from baml_client import b
# from baml_client.types import DeviceReport, RiskLevel
#
# report: DeviceReport = b.AnalyzeDevice(
#     device_id="JNP-001",
#     telemetry="CPU: 94.2%, PFE: 412/hr"
# )
# print(report.risk_level)  # RiskLevel.HIGH
# print(report.risk_level == RiskLevel.HIGH)  # True
"""

print("Generated Python client pattern:")
print(BAML_GENERATED_PYTHON)

---
## 🏭 Part 5 — Production Model Hierarchy
### 🔴 Advanced: Three-Level Model Pattern

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Literal, Optional
from datetime import datetime, timezone
import instructor, anthropic, logging, json

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger("hunter.pipeline")

# ── LEVEL 1: Permissive LLM Output Model ──────────────────────────
# Accept anything the LLM might return (flexible types)
class RawLLMOutput(BaseModel):
    device_id:  str
    risk_level: str            # Accept any string
    score:      float | str    # Accept string or float (LLMs stringify numbers)
    root_cause: Optional[str] = None
    actions:    list[str] = []  # Accept empty list

# ── LEVEL 2: Domain-Validated Model ───────────────────────────────
# Full business logic validation
class DeviceReport(BaseModel):
    device_id:    str = Field(pattern=r"^[A-Z]{2,4}-\d{3}$")
    risk_level:   Literal["LOW", "MED", "HIGH"]
    score:        float = Field(ge=0.0, le=1.0)
    root_cause:   str = Field(min_length=5)
    actions:      list[str] = Field(min_length=1)
    validated_at: datetime = Field(
        default_factory=lambda: datetime.now(timezone.utc)
    )

    @model_validator(mode="after")
    def score_aligns_with_risk(self):
        thresholds = {"LOW": (0.0, 0.5), "MED": (0.3, 0.8), "HIGH": (0.6, 1.0)}
        lo, hi = thresholds[self.risk_level]
        if not (lo <= self.score <= hi):
            raise ValueError(
                f"score {self.score:.2f} inconsistent with {self.risk_level} risk"
            )
        return self

# ── LEVEL 3: API Response Model ────────────────────────────────────
# Serialization-optimized for API responses
class APIDeviceResponse(BaseModel):
    device_id:    str
    risk_level:   str
    score_pct:    str          # "91.0%" instead of 0.91
    summary:      str
    action_count: int
    timestamp:    str          # ISO string

    @classmethod
    def from_domain(cls, report: DeviceReport) -> "APIDeviceResponse":
        return cls(
            device_id=report.device_id,
            risk_level=report.risk_level,
            score_pct=f"{report.score*100:.1f}%",
            summary=f"{report.risk_level} risk: {report.root_cause}",
            action_count=len(report.actions),
            timestamp=report.validated_at.isoformat()
        )

def safe_analyze_device(
    device_id: str,
    telemetry: dict,
    log_path: str = "/tmp/llm_outputs.jsonl"
) -> Optional[APIDeviceResponse]:
    """Production-grade three-level pipeline."""

    client = instructor.from_anthropic(anthropic.Anthropic())

    try:
        # Step 1: Get raw LLM output (permissive)
        raw: RawLLMOutput = client.messages.create(
            model="claude-opus-4-5", max_tokens=1024,
            response_model=RawLLMOutput, max_retries=3,
            messages=[{"role": "user", "content":
                f"Analyze {device_id}: {json.dumps(telemetry)}"
            }]
        )

        # Step 2: Log raw output BEFORE strict validation
        with open(log_path, "a") as f:
            f.write(raw.model_dump_json() + "\n")

        # Step 3: Promote to domain model (strict validation)
        domain = DeviceReport(
            device_id=raw.device_id.upper().strip(),
            risk_level=raw.risk_level.upper().strip(),
            score=float(raw.score),
            root_cause=raw.root_cause or "Unknown cause",
            actions=raw.actions if raw.actions else ["Monitor device"]
        )

        # Step 4: Promote to API response model
        api_response = APIDeviceResponse.from_domain(domain)

        logger.info(f"✓ {device_id}: {domain.risk_level} risk ({domain.score:.2f})")
        return api_response

    except Exception as e:
        logger.error(f"✗ {device_id}: {type(e).__name__}: {e}")
        return None  # Graceful degradation

# Run it
result = safe_analyze_device(
    "JNP-001",
    {"cpu_util": "94.2%", "mem_util": "67.3%", "pfe_errors_per_hr": 412}
)

if result:
    print(f"\n✓ API Response:")
    print(result.model_dump_json(indent=2))

### 🟣 Expert: Pydantic Settings

In [ ]:
# pip install pydantic-settings
from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import Field, SecretStr

class HunterPipelineSettings(BaseSettings):
    """Type-safe settings loaded from environment variables.
    
    Set env vars like:
        HUNTER_S3_BUCKET=marevlo-data
        HUNTER_ANOMALY_THRESHOLD=0.8
    """
    model_config = SettingsConfigDict(
        env_prefix="HUNTER_",     # Reads HUNTER_* env vars
        env_file=".env",          # Also loads from .env file
        env_file_encoding="utf-8",
        case_sensitive=False
    )

    # Required settings (no default)
    # aws_access_key_id:     SecretStr  # HUNTER_AWS_ACCESS_KEY_ID
    # aws_secret_access_key: SecretStr  # HUNTER_AWS_SECRET_ACCESS_KEY
    # s3_bucket:             str        # HUNTER_S3_BUCKET

    # Optional settings with defaults (for demo without env vars)
    aws_region:        str   = "us-east-1"
    model_version:     str   = "v2"
    anomaly_threshold: float = Field(default=0.75, ge=0, le=1)
    lookback_days:     int   = Field(default=7, ge=1, le=90)
    max_workers:       int   = Field(default=4, ge=1, le=32)
    debug:             bool  = False

    @property
    def model_s3_key(self) -> str:
        return f"models/hunter_{self.model_version}.pkl"

# Load from environment (uses defaults for demo)
settings = HunterPipelineSettings()
print("Pipeline settings (from env/defaults):")
print(f"  aws_region:        {settings.aws_region}")
print(f"  model_version:     {settings.model_version}")
print(f"  anomaly_threshold: {settings.anomaly_threshold}")
print(f"  lookback_days:     {settings.lookback_days}")
print(f"  max_workers:       {settings.max_workers}")
print(f"  model_s3_key:      {settings.model_s3_key}")
print(f"  debug:             {settings.debug}")

# Test type validation
try:
    bad = HunterPipelineSettings(anomaly_threshold=1.5)  # > 1.0!
except Exception as e:
    print(f"\nSettings validation caught: {e.errors()[0]['msg']}")

---
## 🏆 Module Challenge

Build a **production-grade multi-device analysis system** using Pydantic that:

1. Defines a `BatchAnalysisRequest` model with:
   - `devices`: list of device IDs (each must match `^[A-Z]{2,4}-\d{3}$`)
   - `time_window_hours`: int (1-168, default=24)
   - `min_severity_filter`: optional Literal["LOW", "MED", "HIGH"]
   - Cross-field validator: if len(devices) > 10, time_window_hours must be ≤ 48

2. Defines a `BatchAnalysisResult` model with:
   - `request_id`: str (auto-generated UUID)
   - `total_devices`: computed field (len of devices)
   - `high_risk_count`: int
   - `reports`: list of DeviceReport (from Part 1)
   - `processed_at`: datetime (auto-set)

3. Implements `analyze_batch(request: BatchAnalysisRequest) -> BatchAnalysisResult` that:
   - Calls Claude via instructor for each device
   - Filters results by min_severity_filter if set
   - Handles failures gracefully (None results skipped)
   - Returns a fully validated BatchAnalysisResult

**Bonus:** Add a `field_serializer` that exports results as JSONL to a file.

In [ ]:
# Your solution here!
import uuid
from pydantic import BaseModel, Field, computed_field, field_validator, model_validator
from typing import Literal, Optional
from datetime import datetime, timezone

# Step 1: Define BatchAnalysisRequest
class BatchAnalysisRequest(BaseModel):
    devices: list[str]
    # Add your fields and validators here
    pass

# Step 2: Define BatchAnalysisResult  
class BatchAnalysisResult(BaseModel):
    request_id: str = Field(default_factory=lambda: str(uuid.uuid4())[:8])
    # Add your fields and computed fields here
    pass

# Step 3: Implement analyze_batch
def analyze_batch(request: BatchAnalysisRequest) -> BatchAnalysisResult:
    # Your implementation here
    pass

# Test it
# request = BatchAnalysisRequest(
#     devices=["JNP-001", "JNP-002", "MX-001"],
#     time_window_hours=24,
#     min_severity_filter="MED"
# )
# result = analyze_batch(request)
# print(result.model_dump_json(indent=2))

---
## 📚 Module Summary

| Concept | Key Tool | Key Pattern |
|---------|----------|-------------|
| **Output validation** | `pydantic.BaseModel` | `model_validate()` for LLM data |
| **Field rules** | `Field(ge=, le=, pattern=)` | Constraint at definition time |
| **Single-field logic** | `@field_validator` | Normalize + validate in one step |
| **Cross-field logic** | `@model_validator(mode="after")` | Business invariants |
| **Computed output** | `@computed_field @property` | Derived fields in JSON output |
| **JSON control** | `@field_serializer` | Custom serialization format |
| **Tool schema** | `model.model_json_schema()` | Pydantic → Anthropic tool |
| **Auto-retry** | `instructor` library | `response_model=YourModel` |
| **BAML** | `baml-cli generate` | DSL → typed Python client |
| **Settings** | `pydantic-settings` | Env vars → typed config |

### Critical Rules
1. **Always use `model_validate()`** (not constructor) for LLM data — it handles coercion
2. **Three-level model hierarchy**: RawLLM → Domain → APIResponse
3. **Log raw LLM output BEFORE strict validation** — preserve debugging data
4. **Graceful degradation**: return None on failure, never crash the batch
5. **`description` = prompt**: write Field descriptions as LLM instructions

---
**Next Module → Module 05: Jinja2 Prompt Templates**